#Notebook Overview
This Test file attempts to implement a different model architecture to improve models accuracy. This implemntation uses different operations such as padding the pixel values to preserve pixel spatial dimensions, kernel initilization set to 'he_uniform' for better weight initializations during training and also BatchNormalization which introduces noise due different variance and mean in different batches, it also results in a smoother and faster gradient descent. Also I have tweaked the number of epochs to 30
# Conclusion

Achieved accuracy of 56% on test data


In [ ]:
try:
  # This command only in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Get project files from internet
!wget https://cdn.freecodecamp.org/project-data/cats-and-dogs/cats_and_dogs.zip

!unzip cats_and_dogs.zip #extracts file

PATH = 'cats_and_dogs'

train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')
test_dir = os.path.join(PATH, 'test')

# Get number of files in each directory. The train and validation directories
# each have the subdirecories "dogs" and "cats".
#counts no of files in each drectory
total_train = sum([len(files) for r, d, files in os.walk(train_dir)])
total_val = sum([len(files) for r, d, files in os.walk(validation_dir)])
total_test = len(os.listdir(test_dir))

# Variables for pre-processing and training.
batch_size = 128
epochs = 15
IMG_HEIGHT = 150
IMG_WIDTH = 150

In [ ]:
print(total_train,total_test,total_val)

In [ ]:
# 3
train_image_generator = ImageDataGenerator(rescale = 1./ 255) #ensures float division is applied
validation_image_generator =  ImageDataGenerator(rescale = 1./ 255)

#Applying transformations
test_image_generator =  ImageDataGenerator(rescale = 1./ 255)

train_data_gen = train_image_generator.flow_from_directory(train_dir,batch_size = batch_size,target_size = (IMG_HEIGHT,IMG_WIDTH), class_mode = 'binary')
val_data_gen = validation_image_generator.flow_from_directory(validation_dir,batch_size = batch_size,target_size = (IMG_HEIGHT,IMG_WIDTH), class_mode = 'binary')


In [ ]:
test_data_gen = test_image_generator.flow_from_directory(test_dir,target_size = (IMG_HEIGHT,IMG_WIDTH))

In [ ]:
'''
flow from directory expects nested directories sincetest folder lacks one the appropriate decision is t point back to the parent then direct it to test
'''
parent_dir = PATH

test_data_gen = test_image_generator.flow_from_directory(parent_dir,classes = ['test'],target_size = (IMG_HEIGHT,IMG_WIDTH),batch_size = batch_size,shuffle = False,class_mode=None)


In [ ]:
#visulizing test data
test_images = os.listdir(test_dir)
file_names = [os.path.join(test_dir,f) for f in test_images]
print(file_names)

In [ ]:
#select random image
import random
x = random.choice(file_names)
print(x)

In [ ]:
from tensorflow.keras.preprocessing.image import load_img,img_to_array
import random
x = load_img(x,target_size = (150,150))
x = img_to_array(x)
x /= 255.0
plt.imshow(x)

In [ ]:
#investigating shape
x.shape

In [ ]:
# 4
def plotImages(images_arr, probabilities = False):
    fig, axes = plt.subplots(len(images_arr), 1, figsize=(5,len(images_arr) * 3))
    if probabilities is False:
      for img, ax in zip( images_arr, axes):
          ax.imshow(img)
          ax.axis('off')
    else:
      for img, probability, ax in zip( images_arr, probabilities, axes):
          ax.imshow(img)
          ax.axis('off')
          if probability > 0.5:
              ax.set_title("%.2f" % (probability*100) + "% dog")
          else:
              ax.set_title("%.2f" % ((1-probability)*100) + "% cat")
    plt.show()

sample_training_images, _ = next(train_data_gen)
plotImages(sample_training_images[:5])


In [ ]:
# 5
train_image_generator = ImageDataGenerator( rescale = 1./255,
rotation_range=15, # images will be randomly rotated by a degree of 15
width_shift_range=0.2,# Images will be randomly shifted horizontally by up to 20% of their total width.
height_shift_range=0.1,#img randomly shifted vertically
shear_range=0.1,# Images will be randomly sheared (tilted along an axis) by up to 20 degrees.
zoom_range=0.2, #Images will be randomly zoomed in by up to 20%
horizontal_flip=True, # randomly flipped
fill_mode='nearest')





In [ ]:
#applying data augmentation
train_data_gen = train_image_generator.flow_from_directory(batch_size=batch_size,
                                                     directory=train_dir,
                                                     target_size=(IMG_HEIGHT, IMG_WIDTH),
                                                     class_mode='binary')

augmented_images = [train_data_gen[0][0][0] for i in range(5)]

plotImages(augmented_images)


# Model Architecture 4
Adding a dropout layer on top of architecture 2

In [ ]:


# 7
#building model architecture
model = Sequential()
model.add(Input(shape = (IMG_HEIGHT,IMG_WIDTH,3))) #input shape

#first convolution layer
model.add(Conv2D((32),(5,5),activation = 'relu',kernel_initializer = 'he_uniform')) #padding to maintain spatial information
model.add(tf.keras.layers.BatchNormalization()) #introducing batchnormalization for stable convergence
model.add(MaxPooling2D((2,2))) #enhances feautures,reduces dimensionality
model.add(Dropout(0.25)) # Adding dropout after first pooling layer

#second convolution layer
model.add(Conv2D((64),(3,3),activation = 'relu',kernel_initializer = 'he_uniform'))
model.add(tf.keras.layers.BatchNormalization()) #introducing batchnormalization for stable convergence
model.add(MaxPooling2D((2,2))) #enhances feautures,reduces dimensionality
model.add(Dropout(0.25)) # Adding dropout after second pooling layer

#third convolution

model.add(Conv2D((64),(3,3),activation = 'relu',kernel_initializer = 'he_uniform'))
model.add(tf.keras.layers.BatchNormalization()) #introducing batchnormalization for stable convergence
#Flatten Layer
model.add(Flatten())
model.add(Dropout(0.5)) # Adding dropout before the dense layers

#Predition Dense Layer

model.add(Dense(128, activation= 'relu',kernel_initializer = 'he_uniform'))

model.add(Dense(1, activation = 'sigmoid')) # 1 neuron as its binary



#model compilation
from tensorflow.keras.optimizers import Adam

model.compile(optimizer = Adam(learning_rate=0.0001),
              loss = 'binary_crossentropy',
              metrics = ['accuracy'])






model.summary()

In [ ]:
#training model
history = model.fit(train_data_gen,
                    steps_per_epoch= int(np.ceil(train_data_gen.samples / batch_size)), # use generator's sample count
                    epochs = 20,
                    validation_data=val_data_gen,
                    validation_steps=int(np.ceil(val_data_gen.samples / batch_size)))

In [ ]:
# 9
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(20)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
probabilities = [model.predict(test_data_gen, steps = int(np.ceil(total_test / batch_size)))]
test_images= next(test_data_gen)
plotImages(test_images,probabilities = probabilities[0])


In [ ]:
# 11
answers =  [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
            1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0,
            1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1,
            1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
            0, 0, 0, 0, 0, 0]

correct = 0

for probability, answer in zip(probabilities[0], answers):
  if round(probability[0]) == answer:
    correct +=1

percentage_identified = (correct / len(answers)) * 100

passed_challenge = percentage_identified >= 63

print(f"Your model correctly identified {round(percentage_identified, 2)}% of the images of cats and dogs.")

if passed_challenge:
  print("You passed the challenge!")
else:
  print("You haven't passed yet. Your model should identify at least 63% of the images. Keep trying. You will get it!")